In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
import joblib, os
os.makedirs('C:/Users/user/dacon/Smart_Logistics/models', exist_ok=True)

train_raw = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test_raw  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout    = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# building_age_years 추가
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'building_age_years',  # 추가
                        'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train_raw.merge(layout_clean, on='layout_id', how='left')
test  = test_raw.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

X = train[feature_cols]
y = train[TARGET]

print(f'feature 수: {len(feature_cols)}')

feature 수: 104


In [2]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))

cat_params = {
    'iterations'   : 199999,
    'learning_rate': 0.03,
    'depth'        : 8,
    'loss_function': 'MAE',
    'eval_metric'  : 'MAE',
    'task_type'    : 'GPU',
    'random_seed'  : 42,
    'verbose'      : 0,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f'── Fold {fold+1} ──')
    X_tr,  y_tr  = X.iloc[tr_idx],  y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostRegressor(**cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    joblib.dump(model, f'C:/Users/user/dacon/Smart_Logistics/models/cat_age_fold{fold+1}.pkl')

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(test[feature_cols]) / 5

    fold_mae = np.mean(np.abs(y_val - model.predict(X_val)))
    print(f'Fold {fold+1} MAE: {fold_mae:.4f}')

oof_mae = np.mean(np.abs(y - oof_preds))
print(f'\nOOF MAE: {oof_mae:.4f}')

── Fold 1 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 1 MAE: 6.5525
── Fold 2 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 2 MAE: 6.6363
── Fold 3 ──


...

Fold 3 MAE: 6.5057
── Fold 4 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 4 MAE: 6.6193
── Fold 5 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 5 MAE: 6.5722

OOF MAE: 6.5772


In [3]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': test_preds
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_v18_cat_age_kfold.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   14.016787
1  TEST_000001                   15.740337
2  TEST_000002                   20.125687
3  TEST_000003                   18.051286
4  TEST_000004                   17.019844
